In [18]:
!pip install fpdf2 PyMuPDF albumentations opencv-python numpy kaggle


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [19]:
import os
import shutil
import kagglehub
import kagglehub.config

# Create data directories
os.makedirs('data/signatures', exist_ok=True)
os.makedirs('data/manual_stamps', exist_ok=True)

# Authenticate with the provided Kaggle PAT
kagglehub.config.set_kaggle_api_token('your_kaggle_pat_here')  

# Download the signature dataset
try:
    print("Attempting to download robinreni/signature-verification-dataset...")
    path = kagglehub.dataset_download('robinreni/signature-verification-dataset')
    print("Downloaded to:", path)
    
    # We copy the contents to our data/signatures folder for processing
    for item in os.listdir(path):
        s = os.path.join(path, item)
        d = os.path.join('data/signatures', item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
    print("Data copied to data/signatures successfully.")
except Exception as e:
    print(f"Failed to download from Kaggle: {e}")
    print("Visit https://www.kaggle.com/datasets/robinreni/signature-verification-dataset to download manually.")

Kaggle credentials set.
Attempting to download robinreni/signature-verification-dataset...
Downloaded to: C:\Users\User\.cache\kagglehub\datasets\robinreni\signature-verification-dataset\versions\2
Data copied to data/signatures successfully.


In [20]:
import os
import glob
import random
import numpy as np
import cv2
import fitz  # PyMuPDF
import logging
from fpdf import FPDF
from pathlib import Path

# basic logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# get image files directly
def get_image_files(directory):
    """find all png and jpg files in a folder recursively."""
    return [str(p) for p in Path(directory).rglob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}]

# set up folders
STAMPS_DIR = 'stamps' 
SIGNATURES_DIR = 'data/signatures'
OUTPUT_IMAGES_DIR = 'dataset/images'
OUTPUT_LABELS_DIR = 'dataset/labels'

os.makedirs(OUTPUT_IMAGES_DIR, exist_ok=True)
os.makedirs(OUTPUT_LABELS_DIR, exist_ok=True)

# grab our images
stamp_files = get_image_files(STAMPS_DIR)
signature_files = get_image_files(SIGNATURES_DIR)

# template for our pdf builder
class ContractPDF(FPDF):
    def __init__(self):
        super().__init__()
        # load arial or tahoma so we can print russian characters
        font_path = "C:/Windows/Fonts/arial.ttf"
        if not os.path.exists(font_path):
             font_path = "C:/Windows/Fonts/tahoma.ttf"
        
        try:
            self.add_font("Arial", style="", fname=font_path)
            self.set_font("Arial", size=12)
        except Exception as e:
            logging.warning(f"couldn't load font, using fallback. {e}")
            self.set_font("helvetica", size=12)
            
        # keep track of where we place stamps and signatures
        self.signature_bboxes = [] # (x, y, w, h) in mm
        self.stamp_bboxes = []     # (x, y, w, h) in mm

    def generate_document(self):
        self.add_page()
        
        # main title
        self.set_font(size=16)
        self.cell(0, 10, "ДОГОВОР ОКАЗАНИЯ УСЛУГ", align="C", new_x="LMARGIN", new_y="NEXT")
        self.ln(5)
        
        # main text
        self.set_font(size=12)
        
        # add dummy text until we get close to the bottom of the page
        while self.get_y() < 230:
            self.multi_cell(0, 8, "Стороны договорились о нижеследующем. Настоящий договор имеет юридическую силу. Стороны обязуются выполнять все предписанные обязанности. Невыполнение обязанностей влечет расторжение договора. ")
            self.ln(2)

        self.ln(10)
        
        # footer with the signature lines
        self.set_font(size=12)
        start_y = self.get_y()
        
        # left signature block
        self.set_xy(10, start_y)
        self.cell(20, 10, "Заказчик: ")
        
        # line up signature on the blank line
        sig1_x = self.get_x() + 2 
        sig1_y = self.get_y() - 3
        self.cell(50, 10, "_________________ /Петров П.П./")
        
        # stamp marker
        self.set_xy(15, start_y + 10)
        self.cell(20, 10, "М.П.")
        stamp1_x = 15
        stamp1_y = start_y + 5
        
        # save the positions for later
        self.signature_bboxes.append((sig1_x, sig1_y, 40, 15))
        self.stamp_bboxes.append((stamp1_x, stamp1_y, 30, 30))
        
        # right signature block
        self.set_xy(110, start_y)
        self.cell(25, 10, "Исполнитель: ")
        
        sig2_x = self.get_x() + 2
        sig2_y = self.get_y() - 3
        self.cell(50, 10, "_________________ /Иванов И.И./")
        
        # stamp marker
        self.set_xy(115, start_y + 10)
        self.cell(20, 10, "М.П.")
        stamp2_x = 115
        stamp2_y = start_y + 5
        
        self.signature_bboxes.append((sig2_x, sig2_y, 40, 15))
        self.stamp_bboxes.append((stamp2_x, stamp2_y, 30, 30))


# turn a pdf into an image
def rasterize_pdf(pdf_buffer, dpi=200):
    doc = fitz.open("pdf", pdf_buffer)
    page = doc[0]
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
    # convert from rgb to bgr for opencv
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    doc.close()
    return img

def mm_to_px(mm_val, dpi=200):
    return int((mm_val * dpi) / 25.4)


# paste an image over the document nicely
def blend_image(doc_roi, asset_path):
    h_roi, w_roi = doc_roi.shape[:2]
    try:
        asset = cv2.imread(asset_path, cv2.IMREAD_UNCHANGED)
        if asset is None: return doc_roi
    except Exception:
        return doc_roi
        
    asset = cv2.resize(asset, (w_roi, h_roi))
    
    # if it's a transparent png
    if asset.shape[2] == 4:
        alpha = asset[:, :, 3] / 255.0
        alpha = np.expand_dims(alpha, axis=-1)
        asset_rgb = asset[:, :, :3].astype(np.float32)
        doc_float = doc_roi.astype(np.float32)
        blended = asset_rgb * alpha + doc_float * (1 - alpha)
        return np.clip(blended, 0, 255).astype(np.uint8)
        
    # normal rgb image (multiply blending so white becomes clear)
    elif asset.shape[2] == 3:
        asset_float = asset.astype(np.float32)
        doc_float = doc_roi.astype(np.float32)
        blended = (doc_float * asset_float) / 255.0
        return np.clip(blended, 0, 255).astype(np.uint8)
        
    # grayscale image
    elif len(asset.shape) == 2:
        asset_color = cv2.cvtColor(asset, cv2.COLOR_GRAY2BGR)
        asset_float = asset_color.astype(np.float32)
        doc_float = doc_roi.astype(np.float32)
        blended = (doc_float * asset_float) / 255.0
        return np.clip(blended, 0, 255).astype(np.uint8)
        
    return doc_roi


# create one dataset sample
def generate_sample(sample_idx):
    if not stamp_files or not signature_files:
        logging.error("missing stamp or signature files, stopping.")
        return
        
    # make the pdf
    pdf = ContractPDF()
    pdf.generate_document()
    pdf_bytes = pdf.output(dest="S") 
    
    # turn into an image
    base_img = rasterize_pdf(pdf_bytes, dpi=200)
    img_h, img_w = base_img.shape[:2]
    
    yolo_bboxes = []
    yolo_labels = []
    
    # add signatures (class 0)
    for x_mm, y_mm, w_mm, h_mm in pdf.signature_bboxes:
        x_px = mm_to_px(x_mm, 200)
        y_px = mm_to_px(y_mm, 200)
        w_px = mm_to_px(w_mm, 200)
        h_px = mm_to_px(h_mm, 200)
        
        # shuffle sizes a bit to make it look realistic
        w_px += random.randint(-5, 5)
        h_px += random.randint(-3, 3)
        
        # make sure we don't go off the edge
        x2_px, y2_px = min(img_w, x_px+w_px), min(img_h, y_px+h_px)
        w_px, h_px = x2_px - x_px, y2_px - y_px
        if w_px <= 0 or h_px <= 0: continue
        
        # paste it
        sig_path = random.choice(signature_files)
        roi = base_img[y_px:y_px+h_px, x_px:x_px+w_px]
        blended = blend_image(roi, sig_path)
        base_img[y_px:y_px+h_px, x_px:x_px+w_px] = blended
        
        # save coordinates in yolo format
        x_center = (x_px + w_px / 2.0) / img_w
        y_center = (y_px + h_px / 2.0) / img_h
        norm_w = w_px / img_w
        norm_h = h_px / img_h
        
        yolo_bboxes.append([x_center, y_center, norm_w, norm_h])
        yolo_labels.append(0)

    # add stamps (class 1)
    for x_mm, y_mm, w_mm, h_mm in pdf.stamp_bboxes:
        x_px = mm_to_px(x_mm, 200)
        y_px = mm_to_px(y_mm, 200)
        w_px = mm_to_px(w_mm, 200)
        h_px = mm_to_px(h_mm, 200)
        
        x2_px, y2_px = min(img_w, x_px+w_px), min(img_h, y_px+h_px)
        w_px, h_px = x2_px - x_px, y2_px - y_px
        if w_px <= 0 or h_px <= 0: continue
        
        # paste it
        stamp_path = random.choice(stamp_files)
        roi = base_img[y_px:y_px+h_px, x_px:x_px+w_px]
        blended = blend_image(roi, stamp_path)
        base_img[y_px:y_px+h_px, x_px:x_px+w_px] = blended
        
        # save coordinates in yolo format
        x_center = (x_px + w_px / 2.0) / img_w
        y_center = (y_px + h_px / 2.0) / img_h
        norm_w = w_px / img_w
        norm_h = h_px / img_h
        
        yolo_bboxes.append([x_center, y_center, norm_w, norm_h])
        yolo_labels.append(1)
        
    # save everything
    if yolo_bboxes:
        try:
            # write the image file
            img_filename = f"synth_doc_{sample_idx:04d}.jpg"
            img_path = os.path.join(OUTPUT_IMAGES_DIR, img_filename)
            cv2.imwrite(img_path, base_img)
            
            # write the label file
            lbl_filename = f"synth_doc_{sample_idx:04d}.txt"
            lbl_path = os.path.join(OUTPUT_LABELS_DIR, lbl_filename)
            with open(lbl_path, "w") as f:
                for bbox, lbl in zip(yolo_bboxes, yolo_labels):
                    f.write(f"{lbl} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")
        except Exception as e:
            logging.error(f"failed to save document {sample_idx}: {e}")

# run the script
logging.info("starting the generator...")

# test with 10 docs just to make sure it works
TOTAL_SAMPLES = 10
for i in range(TOTAL_SAMPLES):
    generate_sample(i)
    if i % 100 == 0:
        print(f"made {i}/{TOTAL_SAMPLES} documents...")

logging.info(f"all done! generated {TOTAL_SAMPLES} documents.")

INFO: starting the generator...
C:\Users\User\AppData\Local\Temp\ipykernel_21736\3574822601.py:175: DeprecationWarning: "dest" parameter is deprecated since v2.2.0 and will be removed in a future release
  pdf_bytes = pdf.output(dest="S")
INFO: maxp pruned
INFO: LTSH dropped
INFO: hdmx dropped
INFO: cmap pruned
INFO: kern dropped
INFO: post pruned
INFO: PCLT dropped
INFO: JSTF dropped
INFO: meta dropped
INFO: DSIG dropped
INFO: GDEF dropped
INFO: GPOS dropped
INFO: GSUB dropped
INFO: glyf pruned
INFO: Added gid0 to subset
INFO: Added first four glyphs to subset
INFO: Closing glyph list over 'glyf': 50 glyphs before
INFO: Glyph names: ['.notdef', 'colon', 'glyph00001', 'glyph00002', 'period', 'slash', 'space', 'underscore', 'uni0410', 'uni0412', 'uni0413', 'uni0414', 'uni0417', 'uni0418', 'uni041A', 'uni041B', 'uni041C', 'uni041D', 'uni041E', 'uni041F', 'uni0420', 'uni0421', 'uni0423', 'uni042F', 'uni0430', 'uni0431', 'uni0432', 'uni0433', 'uni0434', 'uni0435', 'uni0436', 'uni0437', 'un

made 0/10 documents...


INFO: maxp pruned
INFO: LTSH dropped
INFO: hdmx dropped
INFO: cmap pruned
INFO: kern dropped
INFO: post pruned
INFO: PCLT dropped
INFO: JSTF dropped
INFO: meta dropped
INFO: DSIG dropped
INFO: GDEF dropped
INFO: GPOS dropped
INFO: GSUB dropped
INFO: glyf pruned
INFO: Added gid0 to subset
INFO: Added first four glyphs to subset
INFO: Closing glyph list over 'glyf': 50 glyphs before
INFO: Glyph names: ['.notdef', 'colon', 'glyph00001', 'glyph00002', 'period', 'slash', 'space', 'underscore', 'uni0410', 'uni0412', 'uni0413', 'uni0414', 'uni0417', 'uni0418', 'uni041A', 'uni041B', 'uni041C', 'uni041D', 'uni041E', 'uni041F', 'uni0420', 'uni0421', 'uni0423', 'uni042F', 'uni0430', 'uni0431', 'uni0432', 'uni0433', 'uni0434', 'uni0435', 'uni0436', 'uni0437', 'uni0438', 'uni0439', 'uni043A', 'uni043B', 'uni043C', 'uni043D', 'uni043E', 'uni043F', 'uni0440', 'uni0441', 'uni0442', 'uni0443', 'uni0447', 'uni0449', 'uni044B', 'uni044C', 'uni044E', 'uni044F']
INFO: Glyph IDs:   [0, 1, 2, 3, 17, 18, 29, 